In [4]:
import os
import re
import unicodedata
import pandas as pd
from collections import Counter
from rapidfuzz import fuzz
from rapidfuzz.distance import Levenshtein
from tqdm import tqdm

# ==============================
# CONFIG
# ==============================
csv_file = "20250903_Extrait_Constatations_F2.csv"
comment_col = "Commentaire"
desc_col = "Description"

min_occurrences = 3           # minimum frequency to keep a text cluster
threshold_similarity = 90     # fuzzy similarity percentage
max_typo_chars = 4            # Levenshtein distance tolerance
checkpoint_every = 50         # correlation checkpoints
sep = ";"                     # CSV delimiter

# ==============================
# TEXT HELPERS
# ==============================
def normalize_text(s: str) -> str:
    """Normalize unicode, trim spaces, keep accents for FR/DE languages."""
    if pd.isna(s) or not str(s).strip():
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKC", s)
    s = s.replace("\xa0", " ").strip()
    return s

def clean_text_block(text: str) -> str:
    """Lowercase, remove punctuation but keep FR/DE accented characters."""
    if not text:
        return ""
    text = normalize_text(text).lower()
    text = re.sub(r"[^a-zà-öø-ÿœæçß0-9\s-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    words = [w for w in text.split() if len(w) > 1]
    return " ".join(words)

# ==============================
# CLUSTERING HELPERS
# ==============================
def join_variations_with_counts(variants, counter, delim=" | "):
    return delim.join([f"{v} ({counter.get(v,0)})" for v in variants])

def cluster_texts(candidates, counter, label="Text"):
    """Cluster similar texts with fuzzy and typo tolerance."""
    cluster_groups = {}
    for cluster in tqdm(candidates.keys(), desc=f"Fuzzy clustering {label}"):
        found = False
        for rep in list(cluster_groups.keys()):
            sim = fuzz.ratio(cluster, rep)
            edit_d = Levenshtein.distance(cluster, rep)
            if sim >= threshold_similarity or edit_d <= max_typo_chars:
                cluster_groups[rep].append(cluster)
                found = True
                break
        if not found:
            cluster_groups[cluster] = [cluster]
    return cluster_groups

def summarize_clusters(cluster_groups, counter, total):
    rows = []
    for rep, variants in cluster_groups.items():
        cnt = sum(counter.get(v, 0) for v in variants)
        per_mille = round(cnt / total * 1000, 2) if total > 0 else 0
        variations_text = join_variations_with_counts(variants, counter)
        rows.append({
            "Cluster_Representative": rep,
            "Count": cnt,
            "PerMille": per_mille,
            "Variations": variations_text,
        })
    return pd.DataFrame(rows).sort_values(by="Count", ascending=False)

# ==============================
# LOAD AND CLEAN DATA
# ==============================
df = pd.read_csv(csv_file, delimiter=sep, dtype=str)
df.columns = [normalize_text(c) for c in df.columns]

if comment_col not in df.columns or desc_col not in df.columns:
    raise KeyError(f"Columns '{comment_col}' or '{desc_col}' not found. Available: {df.columns.tolist()}")

# Remove rows with NaN or empty text in both columns
df = df.dropna(subset=[comment_col, desc_col])
df = df[(df[comment_col].str.strip() != "") & (df[desc_col].str.strip() != "")]
df = df.reset_index(drop=True)

# ==============================
# CLUSTER COMMENTAIRES
# ==============================
print("\nExtracting full-text Commentaire clusters (ignoring NaN)...")
cleaned_comments = [clean_text_block(t) for t in tqdm(df[comment_col])]
cleaned_comments = [t for t in cleaned_comments if t]
comm_counter = Counter(cleaned_comments)
comm_total = sum(comm_counter.values())
comm_candidates = {k:v for k,v in comm_counter.items() if v >= min_occurrences}
print(f"Found {len(comm_candidates)} Commentaire candidates (≥{min_occurrences})")

comm_clusters = cluster_texts(comm_candidates, comm_counter, label="Commentaire")
comm_summary_df = summarize_clusters(comm_clusters, comm_counter, comm_total)
comm_summary_df.to_csv("commentaire_clusters_summary_fulltext.csv", sep=sep, index=False)

# ==============================
# CLUSTER DESCRIPTIONS
# ==============================
print("\nExtracting full-text Description clusters (ignoring NaN)...")
cleaned_desc = [clean_text_block(t) for t in tqdm(df[desc_col])]
cleaned_desc = [t for t in cleaned_desc if t]
desc_counter = Counter(cleaned_desc)
desc_total = sum(desc_counter.values())
desc_candidates = {k:v for k,v in desc_counter.items() if v >= min_occurrences}
print(f"Found {len(desc_candidates)} Description candidates (≥{min_occurrences})")

desc_clusters = cluster_texts(desc_candidates, desc_counter, label="Description")
desc_summary_df = summarize_clusters(desc_clusters, desc_counter, desc_total)
desc_summary_df.to_csv("description_clusters_summary_fulltext.csv", sep=sep, index=False)

# ==============================
# CORRELATE COMMENTAIRE ↔ DESCRIPTION
# ==============================
print("\nCorrelating Commentaire clusters with Description clusters (row-wise)...")
comment_texts = [clean_text_block(t) for t in df[comment_col]]
desc_texts = [clean_text_block(t) for t in df[desc_col]]

corr_rows = []
for i, (comm_rep, comm_variants) in enumerate(tqdm(comm_clusters.items(), desc="Row-wise correlation")):
    matched_rows = [idx for idx, text in enumerate(comment_texts) if text in comm_variants]
    if not matched_rows:
        continue

    matched_descriptions = [desc_texts[idx] for idx in matched_rows if desc_texts[idx]]
    desc_counts = Counter()
    for desc_text in matched_descriptions:
        for desc_rep, desc_variants in desc_clusters.items():
            if desc_text in desc_variants:
                desc_counts[desc_rep] += 1
                break

    total_corr = sum(desc_counts.values())
    for d_rep, d_cnt in desc_counts.items():
        d_per_mille = round(d_cnt / total_corr * 1000, 2) if total_corr > 0 else 0
        corr_rows.append({
            "Commentaire_Cluster": comm_rep,
            "Commentaire_Count": sum(comm_counter.get(v,0) for v in comm_variants),
            "Commentaire_PerMille": round(sum(comm_counter.get(v,0) for v in comm_variants) / comm_total * 1000, 2),
            "Description_Cluster": d_rep,
            "Description_Count": d_cnt,
            "Description_PerMille": d_per_mille,
        })

    if (i + 1) % checkpoint_every == 0:
        pd.DataFrame(corr_rows).to_csv(f"partial_corr_checkpoint_{i+1}.csv", sep=sep, index=False)
        print(f"Checkpoint {i+1}: {len(corr_rows)} rows saved.")

corr_df = pd.DataFrame(corr_rows)
corr_df.to_csv("commentaire_description_correlations_fulltext.csv", sep=sep, index=False)
print("✅ Saved 'commentaire_description_correlations_fulltext.csv' with", len(corr_df), "rows.")



Extracting full-text Commentaire clusters (ignoring NaN)...


100%|█████████████████████████████████████████████████████████████████████████████| 735/735 [00:00<00:00, 31149.28it/s]


Found 43 Commentaire candidates (≥3)


Fuzzy clustering Commentaire: 100%|███████████████████████████████████████████████████| 43/43 [00:00<00:00, 304.67it/s]



Extracting full-text Description clusters (ignoring NaN)...


...

Found 42 Description candidates (≥3)


Fuzzy clustering Description: 100%|███████████████████████████████████████████████████| 42/42 [00:00<00:00, 218.56it/s]



Correlating Commentaire clusters with Description clusters (row-wise)...


Row-wise correlation: 100%|██████████████████████████████████████████████████████████| 38/38 [00:00<00:00, 2976.79it/s]

✅ Saved 'commentaire_description_correlations_fulltext.csv' with 41 rows.
